In [29]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import cpca
import os
os.chdir("/Users/anora/Library/CloudStorage/Dropbox-TeamMG/Wanru Wu/Cloudseeding_Anora/灾害数据")
df = pd.read_stata("自然灾害事件_since_2014_CN.dta")

# regenerate the hails event
df = df[df['Summary'].str.contains('雹', na=False)]
df = df[['EventStartDate','ProvinceCode','Areas','Province','City']]

# clean the data
df['City'] = df['City'].str.replace("、",",")
df['City'] = df['City'].str.replace("，",",")
df['City'] = df['City'].str.split(',')
df = df.explode('City')

# find the adcode
df.reset_index(drop=True,inplace=True)
df['adcode'] = cpca.transform(df["City"])['adcode']

data = df[~df['adcode'].isna() & (df['ProvinceCode'].astype(str).str[:2] == df['adcode'].astype(str).str[:2])]

In [27]:
data

,EventStartDate,ProvinceCode,Areas,Province,City,adcode
0,2014-04-08 00:00:00,520000,六盘水市盘县、黔西南布依族苗族自治州安龙县和兴义市,贵州省,六盘水市,520200
1,2014-04-08 00:00:00,520000,六盘水市盘县、黔西南布依族苗族自治州安龙县和兴义市,贵州省,黔西南布依族苗族自治州,522300
2,2014-04-09 00:00:00,450000,广西百色、河池,广西壮族自治区,广西,450000
3,2014-04-14 00:00:00,620000,兰州、白银2市5个县（区）,甘肃省,兰州市,620100
4,2014-04-14 00:00:00,620000,兰州、白银2市5个县（区）,甘肃省,白银市,620400
...,...,...,...,...,...,...
3231,2020-09-07 00:00:00,410000,南阳市内乡县、洛阳市宜阳县、安阳市林州市,河南省,洛阳市,410300
3232,2020-09-07 00:00:00,410000,南阳市内乡县、洛阳市宜阳县、安阳市林州市,河南省,安阳市,410500
3233,2020-09-07 00:00:00,150000,赤峰市松山区、敖汉旗,内蒙古自治区,赤峰市,150400
3234,2020-09-23 00:00:00,540000,日喀则、昌都2市5个县(区),西藏自治区,日喀则市,540200


In [28]:

# Drop rows where last four digits of adcode are "0000", except for specified values
mask = (data['adcode'].astype(str).str[-4:] == "0000") & (~data['adcode'].astype(str).isin(["120000", "110000", "500000", "310000"]))
data = data[~mask]
data

,EventStartDate,ProvinceCode,Areas,Province,City,adcode
0,2014-04-08 00:00:00,520000,六盘水市盘县、黔西南布依族苗族自治州安龙县和兴义市,贵州省,六盘水市,520200
1,2014-04-08 00:00:00,520000,六盘水市盘县、黔西南布依族苗族自治州安龙县和兴义市,贵州省,黔西南布依族苗族自治州,522300
3,2014-04-14 00:00:00,620000,兰州、白银2市5个县（区）,甘肃省,兰州市,620100
4,2014-04-14 00:00:00,620000,兰州、白银2市5个县（区）,甘肃省,白银市,620400
5,2014-04-16 00:00:00,520000,遵义市道真仡佬族苗族自治县、务川仡佬族苗族自治县,贵州省,遵义市,520300
...,...,...,...,...,...,...
3231,2020-09-07 00:00:00,410000,南阳市内乡县、洛阳市宜阳县、安阳市林州市,河南省,洛阳市,410300
3232,2020-09-07 00:00:00,410000,南阳市内乡县、洛阳市宜阳县、安阳市林州市,河南省,安阳市,410500
3233,2020-09-07 00:00:00,150000,赤峰市松山区、敖汉旗,内蒙古自治区,赤峰市,150400
3234,2020-09-23 00:00:00,540000,日喀则、昌都2市5个县(区),西藏自治区,日喀则市,540200


In [30]:

# Drop empty start date events
data = data[data['EventStartDate']!=""]

# Generate citycode2 from the first four digits of adcode
data['citycode2'] = data['adcode'].astype(str).str[:4]
#省直辖的单独算一个城市
data.reset_index(drop=True,inplace=True)
data['citycode3'] = data['adcode'].astype(str).str[2:4]
data.loc[data['citycode3'] == "90", 'citycode2'] = data['adcode'].astype(str)
data['citycode2'] = data['citycode2'].astype(int)

# Adjust citycode2 for direct-controlled municipalities
for code, new_code in zip(["11", "12", "31", "50"], [1100, 1200, 3100, 5000]):
    data.loc[data['citycode2'].astype(str).str[:2] == code, 'citycode2'] = new_code

os.chdir("/Users/anora/Library/CloudStorage/Dropbox-TeamMG/Wanru Wu/Cloudseeding_Anora/灾害数据")
data.to_csv("disaster_adcode_hails.csv")
